<div style="text-align: center;">
    <img src="https://mejores.com/wp-content/uploads/2020/08/Universidad-Tecnica-Federico-Santa-Maria.jpg" title="Title text" width="20%" height="20%" />
</div>



<hr style="height:2px;border:none"/>
<h1 align='center'><strong>EIN092B - Visualización de datos con alta dimensionalidad</strong></h1>

**Objetivo:**

Introducir los conceptos fundamentales de la reducción de dimensionalidad y mostrar, de forma práctica, cómo funcionan dos de las técnicas más utilizadas para visualizar datos de alta dimensionalidad: **PCA** y **t-SNE**.

**Motivación:**

- Entender por qué los datos con muchas variables son difíciles de visualizar e interpretar directamente.
- Aprender a reducir esa complejidad conservando la información más relevante.
- Reconocer cuándo conviene usar un método lineal (PCA) o uno no lineal (t-SNE).

Como caso de estudio utilizaremos el dataset **Breast Cancer Wisconsin Diagnostic**, disponible en la librería [scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html), y más adelante el dataset de rostros **Olivetti Faces**, de la misma librería.


## **Datos de alta dimensionalidad**

Cuando hablamos de la **dimensionalidad** de un dataset, nos referimos simplemente al número de variables (columnas o características) que describen cada observación. Por ejemplo, en el dataset que usaremos, cada paciente está descrito por 30 variables distintas, por lo que decimos que los datos "viven" en un espacio de 30 dimensiones.

A la hora de intentar visualizar la relación entre estas variables, nos encontramos con una limitación fundamental: nuestra capacidad de representación visual está restringida a un máximo de tres dimensiones. Esto puede entenderse de forma intuitiva: con una variable, podemos ubicar cada dato en una línea; con dos variables, en un plano, como en un gráfico de dispersión; y con tres variables, todavía podemos imaginar un espacio tridimensional como una nube de puntos.

Sin embargo, al incorporar una cuarta variable o más desde ese punto, dejamos de tener una forma natural de representar directamente los datos. Aunque matemáticamente podemos trabajar con espacios de muchas dimensiones, nuestra percepción visual y nuestra intuición geométrica están limitadas a tres dimensiones, por lo que resulta necesario recurrir a otras estrategias para comprender y visualizar estas relaciones.


A medida que aumenta el número de variables, trabajar con los datos comienza a presentar una serie de dificultades conocidas colectivamente como la **maldición de la dimensionalidad** (*curse of dimensionality*). En términos simples, añadir más dimensiones no solo implica disponer de más información, sino que también hace que el espacio en el que se distribuyen los datos crezca y se vuelva cada vez más difícil de explorar y modelar.

- **Dispersión de los datos:** El volumen del espacio crece rápidamente con cada nueva dimensión. Como consecuencia, incluso conjuntos de datos relativamente grandes pueden quedar dispersos, con observaciones cada vez más aisladas entre sí.

- **Pérdida de significado en las distancias:** En espacios de alta dimensionalidad, la diferencia entre las distancias más pequeñas y más grandes tiende a reducirse. En otras palabras, los puntos pueden comenzar a parecer relativamente igual de lejanos entre sí, dificultando el uso de medidas de distancia para identificar similitudes.

- **Necesidad de mayores volúmenes de datos:** Para representar adecuadamente un espacio de muchas dimensiones se necesitan cantidades de datos cada vez mayores. De lo contrario, gran parte del espacio queda sin observaciones y resulta difícil distinguir patrones representativos.

- **Mayor riesgo de sobreajuste (overfitting):** Cuando existen muchas variables en relación con la cantidad de observaciones disponibles, los modelos tienen más facilidad para aprender patrones accidentales o ruido en lugar de relaciones que realmente se generalicen.

- **Mayor costo computacional:** Más variables también implican una mayor cantidad de información que procesar, lo que puede aumentar el tiempo y los recursos necesarios para explorar los datos, entrenar modelos y realizar ciertos cálculos.


La **reducción de dimensionalidad** busca transformar los datos originales, de muchas variables, en una representación de menor dimensión (típicamente 2 o 3) que conserve la mayor cantidad posible de la estructura o información relevante del dataset original. Además de mitigar los problemas anteriores, esto tiene una ventaja directa: permite visualizar los datos.

En este taller trabajaremos con dos técnicas complementarias:

- **PCA (Principal Component Analysis):** Un método lineal, basado en álgebra lineal, que busca las direcciones donde los datos presentan mayor varianza. Es determinista, rápido, y sus resultados son interpretables en términos de las variables originales.

- **t-SNE (t-distributed Stochastic Neighbor Embedding):** Un método no lineal, pensado específicamente para visualización, que busca preservar las relaciones de vecindad local entre puntos (qué observaciones son similares entre sí), aunque para ello distorsione las distancias globales entre grupos.

## **Configuración inicial**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.datasets import load_breast_cancer
from sklearn.datasets import fetch_olivetti_faces

## **PCA**

**PCA** (*Principal Component Analysis*) busca representar los datos utilizando un nuevo sistema de ejes, denominados **componentes principales**. Estos ejes no corresponden necesariamente a las variables originales, sino que representan nuevas direcciones construidas como combinaciones lineales de ellas. La idea es encontrar aquellas direcciones en las que los datos presentan una mayor variabilidad y utilizarlas para construir una representación más compacta.

El primer componente principal corresponde a la dirección en la que los datos presentan la mayor varianza posible. El segundo debe ser ortogonal a la primera y captura la mayor cantidad de varianza restante. Este proceso continúa con los componentes siguientes, de modo que cada uno explica una parte adicional de la variabilidad de los datos. Como resultado, los primeros componentes suelen concentrar una parte importante de la información contenida originalmente en un gran número de variables.

### **Eigenvalues y Eigenvectors**

Matemáticamente, PCA puede formularse a partir de la **matriz de covarianza** de los datos. A partir de ella se calculan los **eigenvectors** y sus correspondientes **eigenvalues**, resolviendo la siguiente relación:

$$\Sigma v = \lambda v$$

Cada **eigenvector** $v$ representa una dirección en el espacio de los datos y define una componente principal. En otras palabras, indica cómo se combinan las variables originales para formar ese nuevo eje. El **eigenvalue** $\lambda$ asociado, en cambio, indica cuánta varianza presentan los datos en esa dirección.

Los eigenvectors se ordenan según sus eigenvalues, desde el mayor hasta el menor. De esta forma, los primeras componentes son aquellos que capturan una mayor cantidad de la variabilidad presente en los datos. Si nuestro objetivo es reducir la dimensionalidad, podemos conservar únicamente los primeros $k$ componentes y descartar aquellos que aportan relativamente poca información adicional.

En `scikit-learn`, `pca.components_` contiene las direcciones de los componentes principales (una fila por componente), mientras que `pca.explained_variance_` contiene la varianza asociada a cada una de ellas. Para evaluar cuánta información conserva cada componente, podemos utilizar `pca.explained_variance_ratio_` para conocer qué proporción de la varianza total del dataset es explicada por cada componente principal.

### **Carga y preprocesamiento**

Aplicaremos PCA sobre el dataset Breast Cancer Wisconsin. Este conjunto de datos contiene mediciones calculadas a partir de imágenes digitalizadas de muestras de tejido mamario. Cada observación corresponde a un paciente y está descrita mediante 30 variables numéricas, mientras que la variable objetivo indica la clase asociada a cada muestra.

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target
print(X.shape, y.shape)
X.head()

Antes de aplicar PCA, es necesario considerar cuidadosamente la escala de las variables. PCA busca las direcciones de mayor varianza en los datos, por lo que las variables con valores numéricamente más grandes pueden ejercer una influencia desproporcionada sobre el resultado, incluso cuando esa diferencia se debe únicamente a sus unidades de medida.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled.mean()

### **Número de componentes**

Una estrategia habitual para determinar el número de componentes a conservar consiste en observar cómo cambia la varianza explicada a medida que incorporamos más componentes. Para ello podemos construir un gráfico de varianza explicada (o de varianza acumulada) en función del número de componentes y buscar un posible **codo**: un punto a partir del cual agregar nuevos componentes produce ganancias cada vez menores.

La idea es encontrar un equilibrio razonable. Si unas pocas componentes capturan gran parte de la variabilidad de los datos, podemos reducir significativamente la dimensionalidad sin perder demasiada información. En cambio, si necesitamos muchos componentes para alcanzar una proporción elevada de varianza explicada, una representación en pocas dimensiones implicará una pérdida mayor de información.

In [ ]:
n = 5
pca = PCA(n_components=n)
X_pca = pca.fit_transform(X_scaled)

In [ ]:
plt.figure(figsize=(3,2))

explained_var = pca.explained_variance_ratio_
print("Varianza acumulada", round(explained_var.sum(),2))

plt.bar(range(1, len(explained_var)+1), explained_var, edgecolor = 'k', color = 'teal')

plt.xlabel('Componentes principales')
plt.xticks(np.arange(n+1))
plt.ylabel('Varianza explicada')
plt.title('Varianza explicada por cada componente')
plt.show()


### **Visualización en 2D**

Una vez aplicado PCA y transformados los datos, cada observación queda representada mediante sus nuevas coordenadas en el espacio definido por las componentes principales. En nuestro caso, hemos reducido las 30 variables originales a solo 2 componentes, por lo que cada paciente puede representarse como un punto en un plano.

In [ ]:
X_pca.shape, X_scaled.shape, y.shape

Visualizamos los datos primero sin considerar todavía sus etiquetas. Cada punto representa un paciente, y su posición está determinada por sus valores en el primer y segundo componente principal. Esta representación nos permite observar la estructura general de los datos en dos dimensiones. Por ejemplo, podemos identificar concentraciones de observaciones, posibles grupos o regiones en las que los datos se encuentran más dispersos.

In [ ]:
plt.figure(figsize=(6,4))
plt.scatter(X_pca[:,0], X_pca[:,1], alpha=0.7, c = 'skyblue', edgecolor = 'k')
plt.xlabel('Componente principal 1')
plt.ylabel('Componente principal 2')
plt.show()

Para explorar si estas estructuras se relacionan con las clases originales del dataset, podemos utilizar la variable `y` para asignar un color diferente a cada clase. Esto nos permite evaluar si la estructura encontrada por PCA se relaciona, al menos parcialmente, con las diferencias entre las clases conocidas.

In [ ]:
plt.figure(figsize=(6,4))
plt.scatter(X_pca[y==0,0], X_pca[y==0,1], c = 'r', alpha=0.7, label='Benigno', edgecolor = 'k')
plt.scatter(X_pca[y==1,0], X_pca[y==1,1], c = 'b', alpha=0.7, label='Maligno', edgecolor = 'k')
plt.xlabel('Componente principal 1')
plt.ylabel('Componente principal 2')
plt.legend()
plt.show()

### **Interpretación de los loadings**

Cada componente principal es una combinación lineal de las variables originales. Por ejemplo, el primer componente puede expresarse conceptualmente como:

$$PC1 = w_1 \cdot x_1 + w_2 \cdot x_2 + \dots + w_{30} \cdot x_{30}$$


Los coeficientes $w_i$ indican cuánto contribuye cada variable original a la construcción del componente. Nos referimos a estos coeficientes como loadings o cargas. Para cada componente, los coeficientes corresponden a los valores de la dirección encontrada por PCA, es decir, al eigenvector asociado a esa componente. Un loading con un valor absoluto elevado indica que la variable tiene una contribución importante en esa dirección. Por el contrario, un valor cercano a cero indica una contribución relativamente pequeña. El signo, por otro lado, indica si la relación entre la variante y el componente es directa o inversa.

Podemos representar los loadings gráficamente sobre el mismo plano donde visualizamos los datos reducidos. Esto permite observar simultáneamente la distribución de las observaciones y la dirección asociada a algunas de las variables originales. En el siguiente gráfico, seleccionamos las tres variables con mayor valor absoluto de loading en PC1. Cada una se representa mediante una flecha que parte desde el origen. La coordenada horizontal de la flecha corresponde a su loading en PC1, mientras que la coordenada vertical corresponde a su loading en PC2.

In [ ]:
n = 2
pca = PCA(n_components=n)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(6,4))


plt.scatter(X_pca[y==0,0], X_pca[y==0,1], c='r', alpha=0.7, label='Benigno', edgecolor = 'k')
plt.scatter(X_pca[y==1,0], X_pca[y==1,1], c='b', alpha=0.7, label='Maligno', edgecolor = 'k')

plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend()

origin = np.zeros(2)
scaling_factor = 35

loadings = pd.DataFrame(pca.components_.T,
                        columns=['PC1','PC2'],
                        index=data.feature_names)
ordered_pc1 = loadings.reindex(loadings['PC1'].abs().sort_values(ascending=False).index)
#print(loadings)
for feature, row in ordered_pc1.head(3).iterrows():
    pc1 = row['PC1']*scaling_factor
    pc2 = row['PC2']*scaling_factor
    print(feature, pc1, pc2)

    plt.arrow(0, 0, pc1, pc2,
              color='k', width=0.02, head_width=0.5, alpha=0.8,
              label=f'{feature}')
    ang = np.arctan2(pc2, pc1)
    plt.text(pc1 * 1.05, pc2 * 1.05, feature,
            fontsize=9, rotation=ang, rotation_mode='anchor',
            ha='left', va='center',
            bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='none', alpha=0.7))

plt.grid(True)
plt.show()

Es importante destacar que las flechas no representan las variables originales en la misma escala que los puntos: sus valores se multiplican por un `scaling_factor` únicamente para hacerlas visibles sobre el gráfico. Por tanto, su longitud visual no debe interpretarse directamente como una medida cuantitativa de importancia.

Lo más relevante es su dirección y su relación entre sí. Variables que apuntan en direcciones similares presentan contribuciones parecidas en los dos primeros componentes, mientras que variables en direcciones opuestas presentan contribuciones de signo contrario. La importancia relativa de una variable puede analizarse con mayor precisión utilizando los valores numéricos de los loadings.

### **Tabla de loadings**

Una forma más directa de interpretar las componentes consiste en ordenar las variables según el valor absoluto de sus loadings. De esta manera podemos identificar rápidamente cuáles son las características que tienen mayor peso en la construcción de cada componente.

In [ ]:
loadings = pd.DataFrame(pca.components_.T, columns=['PC1','PC2'], index=data.feature_names)
ordered_pc1 = loadings.reindex(loadings['PC1'].abs().sort_values(ascending=False).index)
ordered_pc2 = loadings.reindex(loadings['PC2'].abs().sort_values(ascending=False).index)


In [ ]:
ordered_pc1

In [ ]:
ordered_pc2

Esta información nos permite comenzar a asignar una interpretación concreta a las componentes. Por ejemplo, si entre las variables con mayor loading en PC1 predominan características como el radio, perímetro y área, podríamos interpretar esa componente como una dimensión relacionada principalmente con el tamaño de la lesión. Si otra componente está dominada por variables relacionadas con textura, concavidad o irregularidad, podría representar otro aspecto de las características observadas.

Sin embargo, esta interpretación no debe entenderse como una etiqueta automática. Las componentes principales son construcciones matemáticas y su significado depende del conjunto completo de variables y de sus respectivos coeficientes. Por esta razón, conviene examinar conjuntamente las variables con mayor peso y el contexto del problema antes de asignar una interpretación.

## **t-SNE**

t-SNE (*t-distributed Stochastic Neighbor Embedding*) es una técnica de reducción de dimensionalidad no lineal, diseñada principalmente para la visualización exploratoria de datos en dos o tres dimensiones. A diferencia de PCA, su objetivo no consiste en encontrar las direcciones que explican la mayor cantidad de varianza, sino en preservar, en la medida de lo posible, las relaciones de vecindad entre las observaciones. La intuición central es relativamente simple: si dos puntos son similares y se encuentran cerca en el espacio original de alta dimensionalidad, t-SNE intenta que también aparezcan cerca en la representación de baja dimensión. Esto permite revelar agrupaciones o estructuras locales que pueden resultar difíciles de observar utilizando técnicas lineales.

De manera simplificada, t-SNE construye una representación de las relaciones de vecindad en dos espacios diferentes. Primero, en el espacio original de alta dimensionalidad, el algoritmo estima para cada observación qué tan probable es que las demás observaciones sean sus vecinas. Estas relaciones se modelan mediante probabilidades basadas en la cercanía entre los puntos, utilizando distribuciones gaussianas. Posteriormente, en el espacio de baja dimensionalidad se construye una nueva distribución de probabilidades de vecindad. En este caso se utiliza una distribución t de Student, cuyas colas más pesadas ayudan a abordar el llamado *crowding problem*: la dificultad de representar simultáneamente muchas relaciones de vecindad de un espacio de alta dimensión en solo dos dimensiones. Finalmente, t-SNE ajusta iterativamente las posiciones de los puntos en la representación 2D para que las relaciones de vecindad observadas en ambos espacios sean lo más similares posible. Para ello minimiza una medida de diferencia entre distribuciones conocida como divergencia de *Kullback-Leibler*.

El resultado es un embedding en el que las relaciones locales tienden a preservarse, por lo que las observaciones similares suelen aparecer cerca unas de otras. Esta es una de las principales diferencias con PCA: t-SNE no busca preservar fielmente la geometría global de los datos. Por ello, aunque la cercanía entre puntos dentro de una región suele ser informativa, la distancia entre grupos alejados, su tamaño o los espacios vacíos entre ellos no tienen necesariamente un significado cuantitativo directo. Por ejemplo, que un grupo aparezca dos veces más lejos de otro no significa que las observaciones originales sean dos veces más diferentes. Por tanto, t-SNE debe entenderse principalmente como una herramienta para explorar estructuras locales y posibles agrupaciones, y no como una representación exacta de las distancias originales.

### **Distancias**

Para entender mejor cómo t-SNE construye su representación, podemos observar cómo transforma las distancias entre puntos en probabilidades de vecindad. En el espacio original, una distribución gaussiana permite convertir la distancia entre dos observaciones en una medida de similitud: cuanto más cerca están dos puntos, mayor es la probabilidad asociada a que sean vecinos; a medida que aumenta la distancia, esta probabilidad disminuye rápidamente. De esta forma, en lugar de intentar conservar directamente las distancias originales, t-SNE construye una distribución de probabilidades que representa qué observaciones son más cercanas o lejanas entre sí.

Al trasladar estas relaciones a un espacio de menor dimensión, t-SNE utiliza una distribución t de Student en lugar de una gaussiana. La principal diferencia está en sus colas más pesadas: para una misma distancia, la distribución t asigna una probabilidad mayor que la gaussiana. Esto permite que los puntos que no son vecinos cercanos puedan separarse más en el espacio 2D sin que la representación deje de ser compatible con las relaciones locales originales. Esta propiedad ayuda a mitigar el crowding problem, ya que en dos dimensiones existe mucho menos espacio para representar simultáneamente las relaciones de vecindad presentes en un espacio de alta dimensionalidad.

Finalmente, t-SNE compara las dos distribuciones de probabilidades mediante la divergencia de Kullback-Leibler. Durante la optimización, las posiciones de los puntos se modifican para minimizar esta divergencia. Si dos puntos son vecinos en el espacio original pero aparecen demasiado separados en el mapa, el algoritmo los acerca; si aparecen demasiado cerca pese a no ser vecinos, el algoritmo los separa. El objetivo, por tanto, no es reproducir exactamente las distancias originales, sino encontrar una disposición en baja dimensión que reproduzca lo mejor posible las relaciones de vecindad.

In [ ]:
d = np.linspace(0, 5, 500)
gaussian = np.exp(-(d**2) / 2)
student_t = 1 / (1 + d**2)

plt.figure(figsize=(7, 4))
plt.plot(d, gaussian, label='Gaussiana')
plt.plot(d, student_t, label='t de Student (1 g.l.)')
plt.xlabel('Distancia entre puntos')
plt.ylabel('Similitud / probabilidad relativa')
plt.title('Conversión de distancia en similitud')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
distances = np.array([0.5, 1, 2, 3])
gaussian_values = np.exp(-(distances**2) / 2)
student_t_values = 1 / (1 + distances**2)

comparison = pd.DataFrame({
    'Distancia': distances,
    'Gaussiana': gaussian_values,
    't de Student': student_t_values
})

comparison

### **Hiperparámetros**

El resultado obtenido mediante t-SNE puede cambiar significativamente dependiendo de sus hiperparámetros. Por esto, se recomienda probar con diferentes valores para evaluar las variaciones entre ejecuciones. Los hiperparámetros más importantes son los siguientes:

- **n_components:** Define la dimensionalidad de la representación final. Normalmente utilizamos `n_components=2` para obtener una visualización en un plano, aunque también es posible generar una representación tridimensional.

- **perplexity:** Controla, de forma aproximada, la escala de vecindad que utilizará el algoritmo. Puede interpretarse intuitivamente como una medida relacionada con el número de vecinos efectivos que se consideran al modelar la estructura local de cada punto. Suelen utilizarse valores entre 5 y 50, aunque no existe un valor óptimo universal. Una perplexity baja pone mayor énfasis en estructuras muy locales y puede llegar a fragmentar grupos. Una perplexity más alta considera una vecindad más amplia y puede revelar estructuras de mayor escala, aunque también puede reducir algunos detalles locales.

- **learning_rate:** Este parámetro controla el tamaño de los pasos durante el proceso de optimización. Un valor inadecuado puede dificultar la construcción de una representación útil: valores demasiado bajos pueden hacer que el proceso avance lentamente o produzca estructuras excesivamente compactas, mientras que valores demasiado altos pueden generar una distribución inestable o artificialmente dispersa. En las versiones actuales de scikit-learn, puede utilizarse `learning_rate='auto'`, aunque también es posible especificar manualmente un valor.

- **max_iter:** t-SNE es un algoritmo iterativo, por lo que necesita realizar múltiples pasos de optimización para ajustar las posiciones de los puntos. Si el número de iteraciones es demasiado bajo, el algoritmo puede detenerse antes de haber encontrado una representación estable.

- **random_state:** t-SNE incluye componentes estocásticos, por lo que dos ejecuciones pueden producir representaciones visualmente diferentes. Al fijar su valor, podemos hacer que el resultado sea reproducible.

### **Aplicación**

Ahora aplicaremos t-SNE al mismo dataset utilizado anteriormente. Como las variables originales se encuentran en escalas diferentes, volvemos a estandarizar los datos. Aunque t-SNE no funciona exactamente igual que PCA, las distancias entre observaciones son fundamentales para construir sus relaciones de vecindad, por lo que el escalamiento de las variables sigue siendo un paso importante.

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X_scaled = StandardScaler().fit_transform(X)

tsne = TSNE(n_components=2, random_state=42, perplexity=15, learning_rate=300)
X_tsne = tsne.fit_transform(X_scaled)

plt.figure(figsize=(6,3))
plt.scatter(X_tsne[y==0,0], X_tsne[y==0,1], c='r', alpha=0.6, label='Benigno')
plt.scatter(X_tsne[y==1,0], X_tsne[y==1,1], c='b', alpha=0.6, label='Maligno')

plt.xlabel('t-SNE dimensión 1')
plt.ylabel('t-SNE dimensión 2')
plt.title('t-SNE')
plt.legend()
plt.grid(True)
plt.show()

Al igual que en el caso de PCA, las etiquetas no se utilizan para construir la representación de t-SNE. Se incorporan únicamente después, para colorear los puntos y evaluar visualmente cómo se relaciona la estructura encontrada por el algoritmo con las clases conocidas.

Si observamos una separación parcial entre las clases, esto sugiere que las características originales contienen estructuras relacionadas con la clasificación de las muestras. Sin embargo, debemos evitar interpretar el gráfico como una frontera de clasificación o como una medida exacta de la distancia entre los grupos. La principal información que ofrece t-SNE está en las relaciones locales y la estructura de vecindad que consigue revelar.

### **t-SNE sobre PCA**

Para terminar, aplicaremos t-SNE sobre un dataset donde la alta dimensionalidad resulta mucho más evidente: Olivetti Faces. Este conjunto contiene imágenes en escala de grises de rostros humanos, donde cada imagen tiene una resolución de 64 × 64 píxeles. Si consideramos cada píxel como una variable, cada rostro queda representado originalmente por 4096 características.

In [ ]:
faces = fetch_olivetti_faces()

X = faces.data
y = faces.target

fig, axes = plt.subplots(2, 5, figsize=(8, 3.5))

for _, ax in enumerate(axes.ravel()):
    i = np.random.randint(low=0, high=len(X))
    ax.imshow(X.reshape(X.shape[0], 64,64)[i], cmap='gray')
    ax.set_title(y[i])
    ax.axis("off")

plt.tight_layout
plt.show()

El número que aparece sobre cada imagen corresponde a la identidad del individuo. Podemos observar que pequeñas variaciones en la expresión facial, iluminación, orientación o posición pueden modificar los valores de muchos píxeles simultáneamente. Esto hace que trabajar directamente en las 4096 dimensiones originales sea poco práctico.

#### **PCA como preprocesamiento**

En este contexto, podemos estandarizar los datos y utilizar PCA para reducir las 4096 dimensiones originales a 30 componentes antes de visualizar con t-SNE.

In [ ]:
X_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components=30, random_state=42)
X_pca30 = pca.fit_transform(X_scaled)

tsne = TSNE(n_components=2, random_state=42, perplexity=30, learning_rate=200)
X_tsne = tsne.fit_transform(X_pca30)

plt.figure(figsize=(10,7))
scatter = plt.scatter(X_tsne[:,0], X_tsne[:,1], c=y, cmap='jet', alpha=0.7)
plt.colorbar(scatter, label="Clases (individuos)")
plt.title("t-SNE tras PCA (30 componentes)")
plt.xlabel("t-SNE dimensión 1")
plt.ylabel("t-SNE dimensión 2")
plt.show()

## **Actividad**

Responde a las siguientes preguntas buscando información adicional y razonando críticamente sobre las limitaciones y decisiones de diseño detrás de PCA y t-SNE.

1. En este notebook trabajamos con la matriz de covarianza para obtener los eigenvectors. Investiga: ¿Qué es la Descomposición en Valores Singulares (SVD) y por qué `scikit-learn` la utiliza internamente en PCA en lugar de calcular directamente la matriz de covarianza? ¿Qué ventajas ofrece este enfoque?

2. PCA asume que las direcciones de mayor varianza son las más "informativas". Busca al menos un ejemplo (real o construido) de un dataset donde esta suposición falle. Es decir, donde la varianza no se corresponda con la estructura relevante para el problema (por ejemplo, para separar clases). ¿Por qué ocurre esto?

3. Si en lugar de estandarizar las variables hubiésemos aplicado PCA directamente sobre los datos originales, sin escalar, ¿qué esperarías que ocurriera con los loadings de PC1? Justifica tu respuesta considerando las unidades y escalas de las variables del dataset de cáncer de mama.

4. Investiga qué es la divergencia de Kullback-Leibler y cuál es su rol específico dentro del algoritmo de t-SNE. ¿Por qué se dice que esta divergencia es asimétrica, y qué consecuencia práctica tiene esto sobre qué tipo de errores penaliza más el algoritmo?

5. Ejecuta nuevamente la celda de t-SNE sobre el Breast Cancer Dataset, pero esta vez probando al menos tres valores distintos de `perplexity`, manteniendo el resto de los hiperparámetros fijos. Describe cómo cambia la visualización resultante. ¿En algún caso el resultado parece engañoso o poco representativo de la estructura real de los datos?

6. Busca información sobre **UMAP** (*Uniform Manifold Approximation and Projection*), otra técnica de reducción de dimensionalidad no lineal muy utilizada actualmente. ¿En qué se parece y diferencia conceptualmente a t-SNE?